In [3]:
import os
import urllib.request
import tarfile
from google.colab import drive

# 1. Mount Drive (So we have a place to save the FINAL models)
print("⏳ Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Define Paths
# We download to Colab's temp storage (fast), not Drive (slow)
url = "http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar"
tar_path = "/content/images.tar"
data_dir = "/content/stanford_data"

# 3. Download
print(f"⬇️ Downloading Stanford Dogs Dataset...")
if not os.path.exists(tar_path):
    urllib.request.urlretrieve(url, tar_path)
    print("✅ Download Complete!")

# 4. Extract
print("📂 Extracting images...")
if not os.path.exists(data_dir):
    with tarfile.open(tar_path) as tar:
        tar.extractall(data_dir)
    print("✅ Extracted successfully!")

# 5. Safety Patch for "Truncated Images" (CRITICAL!)
# This prevents the "OSError" crash you saw earlier
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
print("✅ Safety patch applied: Truncated images allowed.")

⏳ Mounting Google Drive...
Mounted at /content/drive
⬇️ Downloading Stanford Dogs Dataset...
✅ Download Complete!
📂 Extracting images...


/tmp/ipython-input-1522597593.py:26: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(data_dir)


✅ Extracted successfully!
✅ Safety patch applied: Truncated images allowed.


In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import json

# 1. Path to extracted images
base_dir = "/content/stanford_data/Images"
SAVE_PATH = "/content/drive/MyDrive/Dog_Project_Stanford_V1"

# Create the folder in your empty Drive
if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)
    print(f"✅ Created folder: {SAVE_PATH}")

# 2. Setup Generators
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

print("🔄 Loading Training Data...")
train_generator = datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

print("🔄 Loading Validation Data...")
val_generator = datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# 3. Save the Breed Map (IMPORTANT)
with open(f"{SAVE_PATH}/class_indices.json", 'w') as f:
    json.dump(train_generator.class_indices, f)

print(f"✅ Map saved to Drive! Detected {train_generator.num_classes} breeds.")

🔄 Loading Training Data...
Found 16508 images belonging to 120 classes.
🔄 Loading Validation Data...
Found 4072 images belonging to 120 classes.
✅ Map saved to Drive! Detected 120 breeds.


In [5]:
from tensorflow.keras.applications import ResNet50, EfficientNetB0, MobileNetV2
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_prep
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobile_prep
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint

EPOCHS = 5
num_classes = train_generator.num_classes
SAVE_PATH = "/content/drive/MyDrive/Dog_Project_Stanford_V1"

def build_and_train(base_arch, name, preprocessor=None):
    print(f"\n🚀 STARTING: {name}...")
    inputs = Input(shape=(224, 224, 3))
    x = preprocessor(inputs) if preprocessor else inputs

    base = base_arch(weights='imagenet', include_top=False, input_tensor=x)
    base.trainable = False

    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    predictions = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=predictions)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

    # Save checkpoint
    ckpt = ModelCheckpoint(f"{SAVE_PATH}/{name}_dogs_finetuned.keras", save_best_only=True, verbose=1)

    model.fit(train_generator, validation_data=val_generator, epochs=EPOCHS, callbacks=[ckpt])
    print(f"✅ {name} Saved!")
    return model

# Train all 3
resnet = build_and_train(ResNet50, "resnet50", resnet_prep)
effnet = build_and_train(EfficientNetB0, "efficientnetb0", None)
mobilenet = build_and_train(MobileNetV2, "mobilenetv2", mobile_prep)


🚀 STARTING: resnet50...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
516/516 ━━━━━━━━━━━━━━━━━━━━ 0s 434ms/step - accuracy: 0.1484 - loss: 3.8884
Epoch 1: val_loss improved from inf to 1.29444, saving model to /content/drive/MyDrive/Dog_Project_Stanford_V1/resnet50_dogs_finetuned.keras
516/516 ━━━━━━━━━━━━━━━━━━━━ 299s 552ms/step - accuracy: 0.1486 - loss: 3.8867 - val_accuracy: 0.6304 - val_loss: 1.2944
Epoch 2/5
516/516 ━━━━━━━━━━━━━━━━━━━━ 0s 411ms/step - accuracy: 0.4870 - loss: 1.7974
Epoch 2: val_loss improved from 1.29444 to 1.04309, saving model to /content/drive/MyDrive/Dog_Project_Stanford_V1/resnet50_dogs_finetuned.keras
516/516 ━━━━━━━━━━━━━━━━━━━━ 265s 514ms/step - accuracy: 0.4870 - loss: 1.7972 - val_accuracy: 0.6906 - val_loss: 1.0431
Epoch 3/5
516/516 ━━━━━━━━━━━━━━━━━━━━ 0s 411ms/step - accuracy: 0.5631 - loss: 1.4804
Epoch 3: val_loss improved from 1.04309 to 1.00943, saving model to /content/drive/MyDrive/Dog_Project_Stanford_V1/resnet50_dogs_finetuned.keras
516/516 ━━━━━━━━━━━━━━━━━━━━ 322s 513ms/step - accuracy: 0.5631 - 

/tmp/ipython-input-2933363379.py:18: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = base_arch(weights='imagenet', include_top=False, input_tensor=x)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
516/516 ━━━━━━━━━━━━━━━━━━━━ 0s 405ms/step - accuracy: 0.3036 - loss: 3.0911
Epoch 1: val_loss improved from inf to 0.98354, saving model to /content/drive/MyDrive/Dog_Project_Stanford_V1/mobilenetv2_dogs_finetuned.keras
516/516 ━━━━━━━━━━━━━━━━━━━━ 285s 526ms/step - accuracy: 0.3039 - loss: 3.0893 - val_accuracy: 0.7031 - val_loss: 0.9835
Epoch 2/5
516/516 ━━━━━━━━━━━━━━━━━━━━ 0s 387ms/step - accuracy: 0.6351 - loss: 1.2667
Epoch 2: val_loss improved from 0.98354 to 0.90307, saving model to /content/drive/MyDrive/Dog_Project_Stanford_V1/mobilenetv2_dogs_finetuned.keras
516/516 ━━━━━━━━━━━━━━━━━━━━ 249s 483ms/step - accuracy: 0.6351 - loss: 1.2666 - val_accuracy: 0.7247 - val_loss: 0.9031
Epoch 3/5
516/516 ━━━━━━━━━━━━━━━━━━━━ 0s 387ms/step - accuracy: 0.6729 - loss: 1.0980
Epoch 3: val_loss improved from 0.90307 to 0.86182, saving model to /content/drive/MyDrive/Dog_Project_Stanford_V1/mobilenetv2_dogs_finetuned.keras
516/516 

In [8]:
import shutil
OUTPUT_FILENAME = "/content/drive/MyDrive/Dog_Models_Backup"
SOURCE_DIR = "/content/drive/MyDrive/Dog_Project_Stanford_V1"

print(f"📦 Zipping models from: {SOURCE_DIR}...")
shutil.make_archive(OUTPUT_FILENAME, 'zip', SOURCE_DIR)
print(f"✅ DONE! Go to your Google Drive and download 'Dog_Models_Backup.zip'")

📦 Zipping models from: /content/drive/MyDrive/Dog_Project_Stanford_V1...
✅ DONE! Go to your Google Drive and download 'Dog_Models_Backup.zip'


In [7]:
import numpy as np
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Model

print("🧠 Training the Manager (Ensemble) Model...")

# 1. Get answers from the 3 Students
print("1/3 Asking ResNet...")
p1 = resnet.predict(val_generator, verbose=1)
print("2/3 Asking EfficientNet...")
p2 = effnet.predict(val_generator, verbose=1)
print("3/3 Asking MobileNet...")
p3 = mobilenet.predict(val_generator, verbose=1)

# 2. Stack their answers
stacked_input = np.concatenate([p1, p2, p3], axis=1)
y_true = tf.keras.utils.to_categorical(val_generator.classes, num_classes)

# 3. Train the Manager to pick the best answer
ensemble_input = Input(shape=(num_classes * 3,))
x = Dense(64, activation='relu')(ensemble_input)
final_output = Dense(num_classes, activation='softmax')(x)

manager_model = Model(inputs=ensemble_input, outputs=final_output)
manager_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
manager_model.fit(stacked_input, y_true, epochs=20, batch_size=32, verbose=0)

# 4. Save it
manager_model.save(f"{SAVE_PATH}/mlp_ensemble.keras")
print("✅ Manager Model Saved!")

🧠 Training the Manager (Ensemble) Model...
1/3 Asking ResNet...
128/128 ━━━━━━━━━━━━━━━━━━━━ 59s 430ms/step
2/3 Asking EfficientNet...
128/128 ━━━━━━━━━━━━━━━━━━━━ 62s 431ms/step
3/3 Asking MobileNet...
128/128 ━━━━━━━━━━━━━━━━━━━━ 58s 416ms/step
✅ Manager Model Saved!


In [9]:
import shutil
OUTPUT_FILENAME = "/content/drive/MyDrive/Dog_Models_Backup"
SOURCE_DIR = "/content/drive/MyDrive/Dog_Project_Stanford_V1"

print(f"📦 Zipping all 4 models from: {SOURCE_DIR}...")
shutil.make_archive(OUTPUT_FILENAME, 'zip', SOURCE_DIR)
print(f"✅ DONE! Go to Google Drive and download 'Dog_Models_Backup.zip'")

📦 Zipping all 4 models from: /content/drive/MyDrive/Dog_Project_Stanford_V1...
✅ DONE! Go to Google Drive and download 'Dog_Models_Backup.zip'


In [11]:
# 1. Install Gradio (The UI Library)
!pip install -q gradio

import gradio as gr
import tensorflow as tf
from tensorflow import keras
import numpy as np
import cv2
from PIL import Image

# 2. Configuration
MODEL_PATH = "/content/drive/MyDrive/Dog_Project_Stanford_V1"

# 3. Load Models
print("⏳ Loading Models...")
models = {
    'resnet50': keras.models.load_model(f"{MODEL_PATH}/resnet50_dogs_finetuned.keras"),
    'efficientnetb0': keras.models.load_model(f"{MODEL_PATH}/efficientnetb0_dogs_finetuned.keras"),
    'mobilenetv2': keras.models.load_model(f"{MODEL_PATH}/mobilenetv2_dogs_finetuned.keras")
}
mlp = keras.models.load_model(f"{MODEL_PATH}/mlp_ensemble.keras")

# Load Class Names
import json
with open(f"{MODEL_PATH}/class_indices.json", 'r') as f:
    class_indices = json.load(f)
class_names = {v: k for k, v in class_indices.items()}

print("✅ Models Loaded!")

# 4. Prediction Logic
def predict_dog(image):
    # Convert to array
    img = np.array(image)
    img = cv2.resize(img, (224, 224))
    img_arr = np.expand_dims(img, axis=0) # No rescale (Smart models handle it)

    # Get predictions
    p1 = models['resnet50'].predict(img_arr, verbose=0)
    p2 = models['efficientnetb0'].predict(img_arr, verbose=0)
    p3 = models['mobilenetv2'].predict(img_arr, verbose=0)

    # Ensemble
    stacked_input = np.concatenate([p1, p2, p3], axis=1)
    final_probs = mlp.predict(stacked_input, verbose=0)[0]

    # Get Top 3
    top_3_indices = final_probs.argsort()[-3:][::-1]
    results = {}
    for i in top_3_indices:
        breed = class_names[i].split('-')[-1].replace('_', ' ').title()
        conf = float(final_probs[i])
        results[breed] = conf

    return results

# 5. Launch App
# 1. Update the Interface logic
interface = gr.Interface(
    fn=predict_dog,
    # We explicitly enable 'webcam' and 'upload' here
    inputs=gr.Image(sources=["upload", "webcam"], type="numpy", label="Upload or Snap Photo"),
    outputs=gr.Label(num_top_classes=3),
    title="🐶 Stanford Dog Classifier",
    description="Click on the 'Webcam' icon below to take a photo."
)

# 2. Launch with debug mode
interface.launch(share=True, debug=True)

⏳ Loading Models...
✅ Models Loaded!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://298409c6b6d220f1b7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1139, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://bcb113e669b6261740.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://298409c6b6d220f1b7.gradio.live


In [1]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import cv2
import numpy as np
import PIL
import io

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      div.style.display = 'flex';
      div.style.flexDirection = 'column';
      div.style.alignItems = 'center';

      const video = document.createElement('video');
      video.style.display = 'block';
      video.style.borderRadius = '10px';
      video.style.marginBottom = '10px';

      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Create a button to capture
      const btn = document.createElement('button');
      btn.textContent = '📸 SNAP PHOTO';
      btn.style.padding = '15px 30px';
      btn.style.fontSize = '18px';
      btn.style.backgroundColor = '#4CAF50';
      btn.style.color = 'white';
      btn.style.border = 'none';
      btn.style.borderRadius = '5px';
      btn.style.cursor = 'pointer';
      div.appendChild(btn);

      // Wait for Click
      await new Promise((resolve) => btn.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])

  # Save to file
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

# --- RUN THE CAMERA ---
print("📸 Opening Camera... Click 'SNAP PHOTO' when ready.")
try:
    filename = take_photo()
    print(f"✅ Photo saved as {filename}")

    # --- RUN PREDICTION IMMEDIATELY ---
    # Load image for AI
    img = cv2.imread(filename)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Fix colors

    # Run your existing prediction logic
    # (Re-using the predict_dog_final function from your previous cell)
    # We pass the numpy array 'img' directly
    label, info = predict_dog_final(img)

    print("\n" + "="*40)
    print(info)
    print("="*40)

    # Show the image
    from matplotlib import pyplot as plt
    plt.imshow(img)
    plt.axis('off')
    plt.show()

except Exception as e:
    print(f"❌ Camera Error: {e}")

📸 Opening Camera... Click 'SNAP PHOTO' when ready.


<IPython.core.display.Javascript object>

✅ Photo saved as photo.jpg
❌ Camera Error: name 'predict_dog_final' is not defined


In [ ]:
import gradio as gr
import tensorflow as tf
from tensorflow import keras
import numpy as np
import cv2
import json
import os
from google.colab import drive

# 1. SETUP
print("⏳ Mounting Drive...")
drive.mount('/content/drive')
MODEL_PATH = "/content/drive/MyDrive/Dog_Project_Stanford_V1"

# 2. LOAD MODELS
print("🧠 Loading AI Models...")
try:
    models = {
        'resnet50': keras.models.load_model(f"{MODEL_PATH}/resnet50_dogs_finetuned.keras"),
        'efficientnetb0': keras.models.load_model(f"{MODEL_PATH}/efficientnetb0_dogs_finetuned.keras"),
        'mobilenetv2': keras.models.load_model(f"{MODEL_PATH}/mobilenetv2_dogs_finetuned.keras")
    }
    mlp = keras.models.load_model(f"{MODEL_PATH}/mlp_ensemble.keras")

    # Load Names
    with open(f"{MODEL_PATH}/class_indices.json", 'r') as f:
        class_indices = json.load(f)
    class_names = {v: k for k, v in class_indices.items()}
    print("✅ System Ready!")
except Exception as e:
    print(f"❌ Error: {e}")

# 3. FULL DATABASE (Traits + Life Span)
# Format: "Life Span | Personality"
breed_db = {
    'Chihuahua': '14-16 years | Sassy, Charming, Graceful. Big personality in a tiny body.',
    'Japanese Spaniel': '10-12 years | Noble, Charming, Loving. An aristocratic lap dog.',
    'Maltese': '12-15 years | Gentle, Playful, Charming. Fearless toy dog with white hair.',
    'Pekinese': '12-14 years | Affectionate, Loyal, Regal. "Lion-like" independence.',
    'Shih Tzu': '10-18 years | Affectionate, Playful, Outgoing. A classic lap warmer.',
    'Blenheim Spaniel': '12-15 years | Affectionate, Gentle, Graceful. Also called Cavalier King Charles.',
    'Papillon': '14-16 years | Happy, Alert, Friendly. Famous for butterfly-like ears.',
    'Toy Terrier': '13-15 years | Spirited, Alert, Intelligent. A tiny watchdog.',
    'Rhodesian Ridgeback': '10-12 years | Dignified, Even-tempered, Affectionate. The African Lion Dog.',
    'Afghan Hound': '12-15 years | Independent, Sweet, Regal. Known for its silky coat.',
    'Basset': '12-13 years | Charming, Patient, Low-key. Famous for long ears and howling.',
    'Beagle': '10-15 years | Merry, Friendly, Curious. Follows its nose everywhere.',
    'Bloodhound': '10-12 years | Independent, Inquisitive, Friendly. The ultimate tracker.',
    'Bluetick': '11-12 years | Smart, Devoted, Tenacious. A nocturnal raccoon hunter.',
    'Black And Tan Coonhound': '10-12 years | Easy-going, Bright, Brave. Mellow at home, rugged on the trail.',
    'Walker Hound': '12-13 years | Smart, Brave, Courteous. A swift, tri-colored hunter.',
    'English Foxhound': '10-13 years | Gentle, Sociable, Athletic. Loves running in a pack.',
    'Redbone': '12-14 years | Even-tempered, Amiable, Eager. A laid-back red hunter.',
    'Borzoi': '9-14 years | Regal, Affectionate, Quiet. A "Greyhound with long hair".',
    'Irish Wolfhound': '6-8 years | Calm, Dignified, Courageous. A gentle giant.',
    'Italian Greyhound': '14-15 years | Alert, Playful, Sensitive. A miniature greyhound.',
    'Whippet': '12-15 years | Affectionate, Playful, Calm. Lightning fast but loves the couch.',
    'Ibizan Hound': '11-14 years | Family-oriented, Polite, Athletic. A deer-like jumper.',
    'Norwegian Elkhound': '12-15 years | Bold, Playful, Dependable. An ancient Viking dog.',
    'Otterhound': '10-13 years | Amiable, Boisterous, Even-tempered. A rare swimmer.',
    'Saluki': '10-17 years | Gentle, Dignified, Independent. The Royal Dog of Egypt.',
    'Scottish Deerhound': '8-11 years | Gentle, Dignified, Polite. The Royal Dog of Scotland.',
    'Weimaraner': '10-13 years | Friendly, Fearless, Obedient. The "Grey Ghost".',
    'Staffordshire Bullterrier': '12-14 years | Clever, Brave, Tenacious. Needs strong leadership.',
    'American Staffordshire Terrier': '12-16 years | Confident, Smart, Good-natured. A powerhouse companion.',
    'Bedlington Terrier': '11-16 years | Loyal, Charming, Frollicking. Looks like a lamb.',
    'Border Terrier': '12-15 years | Affectionate, Plucky, Happy. A tough little worker.',
    'Kerry Blue Terrier': '12-15 years | Alert, Adaptable, People-oriented. Non-shedding coat.',
    'Irish Terrier': '13-15 years | Bold, Dashing, Tenderhearted. The "Daredevil" of dogs.',
    'Norfolk Terrier': '12-16 years | Fearless, Alert, Gregarious. Drop-eared terrier.',
    'Norwich Terrier': '12-15 years | Affectionate, Alert, Curious. Prick-eared terrier.',
    'Yorkshire Terrier': '11-15 years | Sprightly, Tomboyish, Affectionate. Big dog in a small body.',
    'Wire Haired Fox Terrier': '12-15 years | Alert, Confident, Gregarious. A classic hunter.',
    'Lakeland Terrier': '12-16 years | Bold, Zesty, Friendly. A confident little dog.',
    'Sealyham Terrier': '12-14 years | Alert, Outgoing, Funny. The "couch potato" of terriers.',
    'Airedale': '11-14 years | Clever, Confident, Proud. The King of Terriers.',
    'Cairn': '13-15 years | Alert, Cheerful, Busy. Toto from The Wizard of Oz.',
    'Australian Terrier': '11-15 years | Courageous, Spirited, People-oriented. The first Aussie breed.',
    'Dandie Dinmont': '12-15 years | Independent, Smart, Determined. Unique "top-knot" hair.',
    'Boston Bull': '11-13 years | Friendly, Bright, Amusing. The "American Gentleman".',
    'Miniature Schnauzer': '12-15 years | Friendly, Smart, Obedient. A popular family dog.',
    'Giant Schnauzer': '12-15 years | Loyal, Alert, Trainable. A powerful worker.',
    'Standard Schnauzer': '13-16 years | Smart, Fearless, Spirited. The original Schnauzer.',
    'Scotch Terrier': '11-13 years | Confident, Independent, Spirited. The "Diehard".',
    'Tibetan Terrier': '15-16 years | Affectionate, Sensitive, Clever. Not actually a terrier!',
    'Silky Terrier': '13-15 years | Friendly, Quick, Alert. Silkier than a Yorkie.',
    'Soft Coated Wheaten Terrier': '12-14 years | Happy, Steady, Self-Confident. The Irish farm dog.',
    'West Highland White Terrier': '13-15 years | Happy, Loyal, Entertaining. The famous "Westie".',
    'Lhasa': '12-15 years | Confident, Smart, Comical. A Tibetan watchdog.',
    'Flat Coated Retriever': '8-10 years | Cheerful, Optimistic, Good-humored. The "Peter Pan" of dogs.',
    'Curly Coated Retriever': '10-12 years | Confident, Proud, Wickedly Smart. Water dog with curls.',
    'Golden Retriever': '10-12 years | Intelligent, Friendly, Devoted. The classic family dog.',
    'Labrador Retriever': '10-12 years | Active, Friendly, Outgoing. America\'s most popular breed.',
    'Chesapeake Bay Retriever': '10-13 years | Affectionate, Bright, Sensitive. A tough water dog.',
    'German Short Haired Pointer': '10-12 years | Friendly, Smart, Willing. An all-purpose hunter.',
    'Vizsla': '12-14 years | Affectionate, Gentle, Energetic. The "Velcro" dog.',
    'English Setter': '11-15 years | Friendly, Mellow, Merry. The "Gentleman of Dogs".',
    'Irish Setter': '12-15 years | Active, Outgoing, Sweet-natured. Flashy red coat.',
    'Gordon Setter': '12-13 years | Confident, Fearless, Alert. The heaviest setter.',
    'Brittany Spaniel': '12-14 years | Bright, Fun-loving, Upbeat. A versatile bird dog.',
    'Clumber': '10-12 years | Gentle, Loyal, Amusing. The largest spaniel.',
    'English Springer': '12-14 years | Friendly, Playful, Obedient. A classic flushing dog.',
    'Welsh Springer': '12-15 years | Active, Loyal, Affectionate. The red-and-white spaniel.',
    'Cocker Spaniel': '10-14 years | Gentle, Smart, Happy. The smallest sporting dog.',
    'Sussex Spaniel': '13-15 years | Calm, Steady, Affectionate. A low-slung hunter.',
    'Irish Water Spaniel': '12-13 years | Playful, Brave, Smart. The clown of the spaniel family.',
    'Kuvasz': '10-12 years | Protective, Loyal, Patient. A large white guardian.',
    'Schipperke': '13-15 years | Confident, Alert, Curious. The "Little Captain".',
    'Groenendael': '10-14 years | Intelligent, Protective, Intense. The black Belgian Shepherd.',
    'Malinois': '14-16 years | Confident, Smart, Hardworking. The ultimate police dog.',
    'Briard': '12 years | Confident, Smart, Faithful. A shaggy French herder.',
    'Kelpie': '10-13 years | Intelligent, Alert, Eager. An Australian workaholic.',
    'Komondor': '10-12 years | Steady, Fearless, Affectionate. The "Mop Dog".',
    'Old English Sheepdog': '10-12 years | Adaptable, Gentle, Smart. The shaggy dog.',
    'Shetland Sheepdog': '12-14 years | Playful, Energetic, Bright. A mini Lassie.',
    'Collie': '12-14 years | Devoted, Graceful, Proud. Famous for rescuing Timmy.',
    'Border Collie': '12-15 years | Remarkably Smart, Energetic. The world\'s smartest dog.',
    'Bouvier Des Flandres': '10-12 years | Strong-willed, Protective, Gentle. A rugged cattle herder.',
    'Rottweiler': '9-10 years | Loyal, Loving, Confident Guardian. Needs socialization.',
    'German Shepherd': '7-10 years | Confident, Courageous, Smart. A versatile worker.',
    'Doberman': '10-12 years | Alert, Fearless, Loyal. The "Tax Collector\'s Dog".',
    'Miniature Pinscher': '12-16 years | Fearless, Fun-loving, Proud. The "King of Toys".',
    'Greater Swiss Mountain Dog': '8-11 years | Faithful, Family-oriented. A Swiss draft dog.',
    'Bernese Mountain Dog': '7-10 years | Good-natured, Calm, Strong. A gentle giant.',
    'Appenzeller': '12-14 years | Reliable, Fearless, Lively. A Swiss cattle dog.',
    'Entlebucher': '11-13 years | Loyal, Smart, Enthusiastic. The smallest Swiss mountain dog.',
    'Boxer': '10-12 years | Bright, Fun-loving, Active. A muscular athlete.',
    'Bull Mastiff': '7-9 years | Affectionate, Courageous, Docile. A silent guardian.',
    'Tibetan Mastiff': '10-12 years | Independent, Reserved, Intelligent. A primitive guardian.',
    'French Bulldog': '10-12 years | Adaptable, Playful, Smart. Bat ears and a flat face.',
    'Great Dane': '7-10 years | Friendly, Patient, Dependable. The "Apollo of Dogs".',
    'Saint Bernard': '8-10 years | Playful, Charming, Inquisitive. The famous rescue dog.',
    'Eskimo Dog': '10-15 years | Alert, Friendly, Reserved. An Arctic worker.',
    'Malamute': '10-14 years | Affectionate, Loyal, Playful. A powerful sled dog.',
    'Siberian Husky': '12-14 years | Loyal, Outgoing, Mischievous. Born to run.',
    'Affenpinscher': '12-15 years | Confident, Famously Funny. The "Monkey Dog".',
    'Basenji': '13-14 years | Independent, Smart, Poised. The barkless dog.',
    'Pug': '13-15 years | Charming, Mischievous, Loving. A lot of dog in a small space.',
    'Leonberger': '7 years | Bright, Patient, Loving. A lion-like giant.',
    'Newfoundland': '9-10 years | Sweet, Patient, Devoted. A nanny dog.',
    'Great Pyrenees': '10-12 years | Smart, Patient, Calm. A majestic guardian.',
    'Samoyed': '12-14 years | Adaptable, Friendly, Gentle. The "Smiling Sammy".',
    'Pomeranian': '12-16 years | Inquisitive, Bold, Lively. A fluffy extrovert.',
    'Chow': '8-12 years | Dignified, Bright, Serious. Cat-like personality.',
    'Keeshond': '12-15 years | Friendly, Lively, Outgoing. The "Dutch Barge Dog".',
    'Brabancon Griffon': '12-15 years | Sensitive, Alert, Inquisitive. A bearded toy dog.',
    'Pembroke': '12-15 years | Affectionate, Smart, Alert. The Queen\'s favorite.',
    'Cardigan': '12-15 years | Affectionate, Loyal, Smart. The Corgi with a tail.',
    'Toy Poodle': '10-18 years | Agile, Intelligent, Self-Confident. A tiny athlete.',
    'Miniature Poodle': '10-18 years | Active, Proud, Very Smart. A mid-sized athlete.',
    'Standard Poodle': '10-18 years | Active, Proud, Very Smart. A large athlete.',
    'Mexican Hairless': '14-20 years | Loyal, Alert, Cheerful. The Xoloitzcuintli.',
    'Dingo': '15-20 years | Independent, Alert, Wild. The native dog of Australia.',
    'Dhole': '10-13 years | Social, Vocal, Wild. The Asiatic Wild Dog.',
    'African Hunting Dog': '10-12 years | Social, Intense, Wild. The Painted Wolf.'
}

# 4. PREDICTION LOGIC (Smart Matcher)
def predict_dog_final(image):
    if image is None: return "Please upload an image.", ""

    # Preprocess
    img = np.array(image)
    img = cv2.resize(img, (224, 224))
    img_arr = np.expand_dims(img, axis=0)

    # Predict
    p1 = models['resnet50'].predict(img_arr, verbose=0)
    p2 = models['efficientnetb0'].predict(img_arr, verbose=0)
    p3 = models['mobilenetv2'].predict(img_arr, verbose=0)

    # Ensemble Vote
    stacked_input = np.concatenate([p1, p2, p3], axis=1)
    final_probs = mlp.predict(stacked_input, verbose=0)[0]

    # Get Winner
    top_index = final_probs.argmax()
    confidence = final_probs[top_index] * 100

    # 🛠️ FIXED CLEANING: "n02099267-Flat-coated_retriever" -> "Flat Coated Retriever"
    raw_name = class_names[top_index]
    if '-' in raw_name:
        # Split on first dash, keep the rest
        breed_name = raw_name.split('-', 1)[1]
    else:
        breed_name = raw_name

    breed_name = breed_name.replace('_', ' ').title() # Standardize format

    # 🛠️ SMART SEARCH: Find key ignoring case/spaces
    info_found = "Details not found."
    lifespan = "Unknown"
    personality = "Unknown"

    # Create search key (remove spaces, lowercase) e.g., "flatcoatedretriever"
    search_key = breed_name.replace(" ", "").lower()

    for db_key, db_val in breed_db.items():
        if db_key.replace(" ", "").lower() in search_key or search_key in db_key.replace(" ", "").lower():
            breed_name = db_key # Use the pretty dictionary name
            parts = db_val.split('|')
            lifespan = parts[0].strip()
            personality = parts[1].strip()
            break

    # Uncertainty Check
    if confidence < 50.0:
        return {breed_name: confidence / 100}, f"⚠️ UNCERTAIN ({confidence:.1f}%)\n\nI'm not sure, but it looks a little like a {breed_name}."

    # Success Output
    info_text = (
        f"✅ MATCH FOUND!\n\n"
        f"🐕 Breed: {breed_name}\n"
        f"📊 Confidence: {confidence:.1f}%\n"
        f"⏳ Life Span: {lifespan}\n"
        f"📝 Personality: {personality}"
    )

    return {breed_name: confidence / 100}, info_text

# 5. LAUNCH
interface = gr.Interface(
    fn=predict_dog_final,
    inputs=gr.Image(sources=["upload", "webcam"], type="numpy", label="Upload or Snap"),
    outputs=[
        gr.Label(num_top_classes=3, label="Top Predictions"),
        gr.Textbox(label="AI Analysis", lines=6)
    ],
    title="🐶 Ultimate Dog Classifier (120 Breeds)",
    description="Identifies 120 breeds with Lifespan & Personality.",
    theme="default"
)

interface.launch(share=True, debug=True)

⏳ Mounting Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🧠 Loading AI Models...
✅ System Ready!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://af83ac131f2f2ae645.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
